# LLM Classification with Reasoning Model (LM Studio)
Questo notebook campiona 20 record per ciascuna delle 10 classi del dataset `pwc_distilled_balanced.csv` generato precedentemente e li classifica utilizzando un modello reasoning locale (es. gpt-oss 20B) tramite LM Studio.

La particolarità di questo approccio è la gestione dell'output del modello reasoning: il modello tipicamente genera dei token di ragionamento (es. all'interno di tag `<think>...</think>`) prima di fornire la risposta finale. Il codice si occuperà di estrarre solo la classificazione finale scartando il ragionamento.

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import re
from openai import OpenAI
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path

# Configurazione stili grafici
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['figure.dpi'] = 100

## 1. Caricamento e Campionamento dei Dati

In [ ]:
# Caricamento dataset distillato bilanciato
DATA_DIR = Path('../data/distilled')
df_full = pd.read_csv(DATA_DIR / 'pwc_distilled_balanced.csv')

print(f"Dataset completo: {df_full.shape}")

# Campionamento di 20 record per classe
SAMPLES_PER_CLASS = 20
RANDOM_SEED = 42

df_sample = df_full.groupby('Label').sample(n=SAMPLES_PER_CLASS, random_state=RANDOM_SEED).reset_index(drop=True)
df_sample['original_idx'] = df_sample.index  # Salviamo un id locale

print(f"\nDataset campionato: {df_sample.shape}")
print(f"\nDistribuzione classi nel sample:")
print(df_sample['Label'].value_counts())

display(df_sample.head(3))

## 2. Configurazione LM Studio

In [ ]:
# Inizializzazione del client OpenAI per puntare a LM Studio
# LM Studio espone le API compatibili con OpenAI su http://localhost:1234/v1
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# Verifica connessione e modelli
try:
    models = client.models.list()
    print("Modelli disponibili su LM Studio:")
    for m in models.data:
        print(f"- {m.id}")
    # Utilizziamo il primo modello disponibile o quello che decidi di caricare (es. gpt-oss 20B)
    SUPPORTED_MODEL = models.data[0].id if models.data else "gpt-oss-20b-reasoning"
    print(f"\nModello selezionato: {SUPPORTED_MODEL}")
except Exception as e:
    print(f"Errore di connessione a LM Studio: {e}\nAssicurati che il server sia avviato su localhost:1234")
    SUPPORTED_MODEL = "gpt-oss-20b-reasoning"

## 3. Prompt Engineering e Parsing Output Reasoning
I modelli reasoning come DeepSeek-R1 (o simili derivati) spesso usano un formato del tipo:
```
<think>
Il testo parla di X, quindi potrebbe essere Y...
</think>
La categoria finale è Z.
```
La nostra funzione `classify_record` utilizzerà una Regex per rimuovere il contenuto tra i tag `<think>` ed estrarre la risposta vera e propria.

In [ ]:
ALLOWED_CATEGORIES = [
    "Automotive & UVs",
    "Enterprise",
    "Environment",
    "Fintech and Marketing",
    "Generic use",
    "Healthcare AI",
    "Media & Entertainment",
    "Research",
    "Robotics and Industry",
    "Virtual assistants"
]

def create_prompt(description):
    categories_list = "\n".join([f"- {c}" for c in ALLOWED_CATEGORIES])
    system_prompt = f"""You are a text classifier. Classify the following scientific paper description into EXACTLY ONE of these categories:

{categories_list}

CRITICAL RULES:
1. You are a reasoning model. You may output your reasoning first. If you do, put it inside <think>...</think> tags.
2. After the reasoning, you MUST output ONLY the exact category name from the list above. Do not add any other text outside the <think> tags.
"""
    
    user_prompt = f"""Classify this abstract:
"{description}"

Category:"""
    
    return system_prompt, user_prompt

def extract_label_from_reasoning(response_text):
    """
    Rimuove i token di ragionamento (es. <think>...</think>) 
    ed estrae l'etichetta finale.
    """
    # Rimuovi il contenuto del tag <think> (inclusi multiline)
    cleaned_text = re.sub(r'<think>.*?</think>', '', response_text, flags=re.DOTALL)
    
    # Pulizia finale
    cleaned_text = cleaned_text.strip().replace('\n', ' ').strip('\'"')
    
    # Ricerca esatta o substring (case-insensitive)
    result_lower = cleaned_text.lower()
    for cat in ALLOWED_CATEGORIES:
        if cat.lower() == result_lower:
            return cat
            
    # Se non c'è match esatto, cerca come substring
    matches = [cat for cat in ALLOWED_CATEGORIES if cat.lower() in result_lower]
    if matches:
        # Restituisce il match più lungo per evitare conflitti
        return max(matches, key=len)
        
    return f"UNKNOWN_{cleaned_text[:30]}"

def classify_record(description, max_retries=3):
    system_p, user_p = create_prompt(description)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=SUPPORTED_MODEL,
                messages=[
                    {"role": "system", "content": system_p},
                    {"role": "user", "content": user_p}
                ],
                temperature=0.0,
                max_tokens=1000 # Consentiamo più token per includere il ragionamento
            )
            
            raw_response = response.choices[0].message.content
            final_label = extract_label_from_reasoning(raw_response)
            
            return final_label, raw_response
            
        except Exception as e:
            print(f"Errore al tentativo {attempt + 1}: {e}")
            time.sleep(2)
            
    return "ERROR_MAX_RETRIES", ""


## 4. Esecuzione Classificazione

In [ ]:
out_path = '../data/processed/llm_predictions_reasoning.csv'
os.makedirs(os.path.dirname(out_path), exist_ok=True)

buffer = []
BATCH_SAVE = 10

print(f"Inizio classificazione per {len(df_sample)} record...")

for i, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Inferenza Reasoning"):
    # Tronchiamo a 1500 caratteri per evitare contesto troppo lungo se le descrizioni sono enormi
    desc = str(row['description'])[:1500]
    
    llm_label, raw_response = classify_record(desc)
    
    buffer.append({
        'original_idx': row['original_idx'],
        'description': row['description'],
        'true_label': row['Label'],
        'llm_label': llm_label,
        'raw_response': raw_response
    })
    
    if len(buffer) >= BATCH_SAVE:
        # Salva incrementalmente
        df_batch = pd.DataFrame(buffer)
        if not os.path.exists(out_path):
            df_batch.to_csv(out_path, index=False)
        else:
            df_batch.to_csv(out_path, mode='a', header=False, index=False)
        buffer = []

# Salva il rimanente
if buffer:
    df_batch = pd.DataFrame(buffer)
    if not os.path.exists(out_path):
        df_batch.to_csv(out_path, index=False)
    else:
        df_batch.to_csv(out_path, mode='a', header=False, index=False)

print(f"\nClassificazione completata! Risultati salvati in: {out_path}")

## 5. Valutazione e Analisi dei Risultati

In [ ]:
df_results = pd.read_csv(out_path)

# Rimuoviamo eventuali duplicati nel caso in cui la cella precedente sia stata interrotta e ripresa
df_results = df_results.drop_duplicates(subset='original_idx', keep='last')

matched = (df_results['true_label'] == df_results['llm_label']).sum()
total = len(df_results)
print(f"Accuracy complessiva (Reasoning Model): {matched}/{total} ({matched/total*100:.2f}%)\n")

unknown_preds = df_results[df_results['llm_label'].str.startswith('UNKNOWN_', na=False)]
error_preds = df_results[df_results['llm_label'] == 'ERROR_MAX_RETRIES']
print(f"Errori di parsing (UNKNOWN_...): {len(unknown_preds)}")
print(f"Errori di connessione a LM Studio: {len(error_preds)}\n")

print("Accuracy per classe:")
for cat in ALLOWED_CATEGORIES:
    mask = df_results['true_label'] == cat
    if mask.sum() > 0:
        cat_acc = (df_results.loc[mask, 'true_label'] == df_results.loc[mask, 'llm_label']).mean()
        print(f"  {cat:<25} {cat_acc*100:>5.1f}% (n={mask.sum()})")

In [ ]:
valid_mask = df_results['llm_label'].isin(ALLOWED_CATEGORIES)
df_valid = df_results[valid_mask]

if len(df_valid) > 0:
    print(f"\nClassification Report ({len(df_valid)} record correttamente formattati):\n")
    print(classification_report(df_valid['true_label'], df_valid['llm_label'], 
                                labels=ALLOWED_CATEGORIES, zero_division=0))

    # Disegno della matrice di confusione
    cm = confusion_matrix(df_valid['true_label'], df_valid['llm_label'], labels=ALLOWED_CATEGORIES)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=ALLOWED_CATEGORIES, yticklabels=ALLOWED_CATEGORIES, cbar=False)
    plt.xlabel('Predizione (LLM Reasoning)', fontsize=12)
    plt.ylabel('Verità', fontsize=12)
    plt.title('Matrice di Confusione - LLM Classification Reasoning', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Nessun record valido trovato per la matrice di confusione.")

### Esempio di output Raw (incluso ragionamento)

In [ ]:
# Vediamo un esempio di come ha ragionato il modello e come è stato pulito
if len(df_results) > 0:
    sample_row = df_results.iloc[0]
    print("--- ESEMPIO DI RISPOSTA RAW (INCLUSA PARTE DI REASONING) ---")
    print(sample_row['raw_response'])
    print("\n--- OUTPUT ESTRATTO DAL PARSER ---")
    print(sample_row['llm_label'])
    print("\n--- LABEL CORRETTA ---")
    print(sample_row['true_label'])